# DeepSDF: Gaussian vs Diffusion in the Small-Data Regime

Colab driver for the final project. This notebook is a **thin wrapper**: it clones the repo, installs dependencies, mounts Drive for persistence, and calls `scripts/run_grid.py` and `scripts/make_figures.py`. All logic lives in `src/`.

Recommended runtime: **GPU (L4 / A100 on Colab Pro)**.

## 1. Setup: clone, install, mount Drive

In [ ]:
import os
from pathlib import Path

REPO = Path('/content/deepsdf-generative-latent')
CODE = REPO / 'code'

if not REPO.exists():
    !git clone https://github.com/amitbe711/deepsdf-generative-latent.git
else:
    print('Repo already cloned, skipping git clone.')

os.chdir(CODE)
print('Working directory:', os.getcwd())
assert (CODE / 'requirements.txt').exists(), 'requirements.txt not found — check clone path'

!pip install -q -r requirements.txt

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/deepsdf_generative/outputs/grid'
FIG_DIR = '/content/drive/MyDrive/deepsdf_generative/figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# Ensure we stay in the code directory for later cells
os.chdir('/content/deepsdf-generative-latent/code')
print('OUTPUT_DIR:', OUTPUT_DIR)
print('CWD:', os.getcwd())

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 2. Run the N x D grid

Configs (same code path, different scale):
- `configs/base.yaml` - original sweep: N in {10,50,150}, D in {16,32}, generators = gaussian + ddpm.
- `configs/wide.yaml` - **wider study**: N in {10,50,150,300,500}, adds a **GMM** generator (single Gaussian -> GMM -> DDPM) and more Stage-1 iterations. Use this on Colab Pro / Databricks GPU.
- `configs/smoke.yaml` - fast CPU check.

The run is resumable: each cell checkpoints to Drive.

In [ ]:
# Wider study (N up to 500, adds GMM). Swap to configs/base.yaml for the original sweep.
!python scripts/run_grid.py --config configs/wide.yaml --output "$OUTPUT_DIR"

## 3. Regenerate tables and figures for the report

In [ ]:
!python scripts/make_figures.py --input "$OUTPUT_DIR" --figures "$FIG_DIR"

from IPython.display import Image, display
# degradation_generation.png now includes the valid_ratio panel and the GMM curves.
for name in ['degradation_generation.png', 'degradation_reconstruction.png', 'loss_curves.png', 'gallery.png']:
    display(Image(filename=f'{FIG_DIR}/{name}'))

## 4. (Optional) Use a real ShapeNet chairs subset

Upload a folder of chair meshes to Drive, then edit `configs/base.yaml`:

```yaml
data:
  source: mesh_dir
  mesh_dir: /content/drive/MyDrive/shapenet_chairs
```

Inspect the directory first:

In [ ]:
# !python scripts/prepare_data.py --inspect-dir /content/drive/MyDrive/shapenet_chairs